In [1]:
import pandas as pd

In [3]:
df1 = pd.read_excel(r"C:\Users\m.olshanskiy\Desktop\Продажи_49472.xlsx", sheet_name='По корпусам')

In [4]:
df2 = pd.read_excel(r"C:\Users\m.olshanskiy\Desktop\Продажи_49472.xlsx", sheet_name='По проектам')

In [6]:
group_cols = ['Название проекта', 'Месяц_год', '№ проектной декларации']

result_parts = []

for _, group in df.groupby(group_cols):

    # если в группе больше одного корпуса → это дубликаты
    if group['Id дом.рф'].nunique() > 1:

        new_row = group.iloc[[0]].copy()

        # объединяем корпуса
        корпуса = sorted(group['Корпус'].astype(str).unique())
        new_row['Корпус'] = ', '.join(корпуса)

        # объединяем Id дом.рф
        ids = sorted(group['Id дом.рф'].astype(str).unique())
        new_row['Id дом.рф'] = ', '.join(ids)

        result_parts.append(new_row)

    else:
        # если корпус один — ничего не меняем
        result_parts.append(group)

df1 = pd.concat(result_parts, ignore_index=True)

NameError: name 'df' is not defined

In [7]:
group_cols = ['Название проекта', 'Месяц_год']

sum_cols = [
    'Жилые помещения, количество договоров',
    'Жилые помещения, площадь объектов',
    'Жилые помещения, суммарная цена договоров',
    'Нежилые помещения, количество договоров',
    'Нежилые помещения, площадь объектов',
    'Нежилые помещения, суммарная цена договоров',
    'Машино-места, количество договоров',
    'Машино-места, площадь объектов',
    'Машино-места, суммарная цена договоров',
    'Квартиры количество месяц',
    'Квартиры площадь месяц',
    'Квартиры сумма месяц',
    'Нежилые количество месяц',
    'Нежилые площадь месяц',
    'Нежилые сумма месяц',
    'Машино-места количество месяц',
    'Машино-места площадь месяц',
    'Машино-места сумма месяц'
]

# NaN → 0 только в суммируемых колонках
df1[sum_cols] = df1[sum_cols].fillna(0)

# автоматически определяем остальные столбцы
other_cols = [col for col in df1.columns if col not in sum_cols]

# для них берём первое значение
agg_dict = {col: 'first' for col in other_cols}
agg_dict.update({col: 'sum' for col in sum_cols})

df2 = (
    df1
    .groupby(group_cols, as_index=False)
    .agg(agg_dict)
)


KeyError: "['Квартиры количество месяц', 'Квартиры площадь месяц', 'Квартиры сумма месяц', 'Нежилые количество месяц', 'Нежилые площадь месяц', 'Нежилые сумма месяц', 'Машино-места количество месяц', 'Машино-места площадь месяц', 'Машино-места сумма месяц'] not in index"

In [16]:
cols = [
'Площадь жилых помещений',
'Площадь нежилых помещений',
'Площадь жилых и нежилых помещений',
'Количество жилых помещений',
'Количество нежилых помещений',
'Количество машино-мест'
]

df1[cols] = (
    df1.groupby('Id дом.рф')[cols]
      .transform(lambda x: x.ffill().bfill())
)

In [18]:
cols = [
'Площадь жилых помещений',
'Площадь нежилых помещений',
'Площадь жилых и нежилых помещений',
'Количество жилых помещений',
'Количество нежилых помещений',
'Количество машино-мест'
]

df2[cols] = (
    df2.groupby('Id дом.рф')[cols]
      .transform(lambda x: x.ffill().bfill())
)

In [19]:
# Сохранение в один Excel файл на разные листы
with pd.ExcelWriter(r"C:\Users\m.olshanskiy\PycharmProjects\ndv_parsing\НашДомРФ\Продажи\Продажи_Калининград3.xlsx", engine='openpyxl') as writer:
    df1.to_excel(writer, sheet_name='По корпусам', index=False)
    df2.to_excel(writer, sheet_name='По проектам', index=False)